# Kapanin — Analisis dan rekomendasi awal

Notebook ini memakai tiga CSV toko 1 untuk:
1. memeriksa kualitas data;
2. menganalisis penjualan dan diskon;
3. membuat data **produk × minggu**;
4. mengevaluasi baseline prediksi penjualan;
5. membuat contoh status **normal / monitor / discount**.

**Batas data:** seluruh sumber benchmark adalah data sintetis. Baris `generated` sampai September 2026 juga disimulasikan dari pola sumber. `stock_on_hand` hanya berlaku pada `stock_as_of_date`, sehingga tidak boleh dipakai sebagai fitur stok pada minggu historis. Besaran diskon di akhir notebook berasal dari aturan prototipe, belum dari estimasi efek diskon.

## Menjalankan notebook

Jalankan dari folder `Kapanin` atau root workspace untuk memakai CSV lokal. Jika folder itu tidak ada (misalnya di Colab), notebook otomatis membaca tiga CSV dari branch `main` repo GitHub Kapanin. Pastikan `data/store_1/stores.csv`, `products.csv`, dan `transactions.csv` sudah ada di branch `main` dan repositorinya dapat diakses publik. Kebutuhan inti: Python, Pandas, dan NumPy. Grafik memakai Matplotlib bila tersedia.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

candidates = [
    Path.cwd() / "data" / "store_1",
    Path.cwd() / "Kapanin" / "data" / "store_1",
    Path.cwd().parent / "data" / "store_1",
]
required_files = ("stores.csv", "products.csv", "transactions.csv")
DATA_DIR = next((path for path in candidates if all((path / name).exists() for name in required_files)), None)
RAW_BASE_URL = "https://raw.githubusercontent.com/faizwibawa/Kapanin/main/data/store_1"

def csv_source(filename):
    return DATA_DIR / filename if DATA_DIR is not None else f"{RAW_BASE_URL}/{filename}"

print("Sumber data:", DATA_DIR.resolve() if DATA_DIR is not None else RAW_BASE_URL)

In [ ]:
stores = pd.read_csv(csv_source("stores.csv"), dtype={"store_id": "string"})
products = pd.read_csv(
    csv_source("products.csv"),
    dtype={"product_id": "string", "store_id": "string", "sku": "string"},
    parse_dates=["stock_as_of_date", "stock_received_date", "expiry_date"],
)
transactions = pd.read_csv(
    csv_source("transactions.csv"),
    dtype={"transaction_id": "string", "store_id": "string", "product_id": "string"},
    parse_dates=["transaction_date"],
)
for name, frame in [("stores", stores), ("products", products), ("transactions", transactions)]:
    print(f"{name}: {len(frame):,} rows, {len(frame.columns)} columns")
print("Rentang transaksi:", transactions.transaction_date.min().date(), "s.d.", transactions.transaction_date.max().date())
print("Asal data:", transactions.data_origin.value_counts().to_dict())

## 1. Validasi data

Kita cek kunci, hubungan antardata, harga, diskon, dan tanggal. Pemeriksaan ini harus lulus sebelum data dipakai untuk eksperimen.

In [ ]:
assert len(stores) == 1
assert stores.store_id.is_unique and products.product_id.is_unique
assert products.sku.is_unique and transactions.transaction_id.is_unique
assert not stores.isna().any().any()
assert not products.isna().any().any()
assert not transactions.isna().any().any()
assert set(products.store_id).issubset(set(stores.store_id))
assert set(transactions.store_id).issubset(set(stores.store_id))
assert set(transactions.product_id).issubset(set(products.product_id))
assert (products.list_price > products.unit_cost).all()
assert (products.stock_on_hand >= 0).all()
assert (transactions.quantity > 0).all()
assert (transactions.unit_sale_price <= transactions.unit_list_price).all()
assert (transactions.discount_rate.between(0, 1)).all()
assert np.allclose(
    transactions.discount_rate,
    1 - transactions.unit_sale_price / transactions.unit_list_price,
    atol=1e-6,
)
assert ((transactions.discount_rate == 0) == (transactions.promotion_type == "no_discount")).all()
expiry_by_product = products.set_index("product_id").expiry_date
assert (
    transactions.transaction_date
    <= transactions.product_id.map(expiry_by_product)
).all()
print("Semua pemeriksaan kualitas data lulus.")

## 2. Eksplorasi singkat

Pisahkan baris `imported` dan `generated` saat membaca grafik atau metrik. Kenaikan atau penurunan pada data `generated` mencerminkan aturan generator, bukan perubahan pasar yang teramati.

In [ ]:
monthly_sales = (
    transactions.assign(month=transactions.transaction_date.dt.to_period("M").astype(str))
    .groupby(["month", "data_origin"], as_index=False)
    .agg(units_sold=("quantity", "sum"), revenue=("unit_sale_price", "sum"))
)
print("Penjualan bulanan terakhir:")
print(monthly_sales.tail(18).to_string(index=False))
print("\nKategori produk terbesar:")
print(products.category.value_counts().head(10).to_string())
print("\nPorsi transaksi berdiskon menurut asal data:")
print(transactions.assign(has_discount=transactions.discount_rate > 0)
      .groupby("data_origin").has_discount.mean().round(3).to_string())

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("Matplotlib belum tersedia; tabel bulanan di atas tetap dapat dipakai.")
else:
    pivot = monthly_sales.pivot(index="month", columns="data_origin", values="units_sold").fillna(0)
    ax = pivot.plot(figsize=(12, 4), title="Unit terjual per bulan menurut asal data")
    ax.set_xlabel("Bulan")
    ax.set_ylabel("Unit")
    plt.xticks(rotation=60)
    plt.tight_layout()
    plt.show()

## 3. Data produk × minggu

Gunakan hanya baris `imported` untuk evaluasi baseline. Minggu pertama dan terakhir yang tidak penuh dibuang. Minggu tanpa transaksi diberi `units_sold = 0` sebagai **penjualan tercatat nol**, dengan catatan ketersediaan stok pada minggu tersebut tidak diketahui. Produk dimasukkan mulai minggu penjualan pertamanya.

In [ ]:
historical = transactions.loc[transactions.data_origin == "imported", [
    "product_id", "transaction_date", "quantity"
]].copy()
historical["week_start"] = historical.transaction_date.dt.to_period("W-SUN").dt.start_time
first_week = historical.transaction_date.min().to_period("W-SUN").start_time + pd.Timedelta(weeks=1)
last_week = historical.transaction_date.max().to_period("W-SUN").start_time - pd.Timedelta(weeks=1)
weeks = pd.date_range(first_week, last_week, freq="W-MON")
weekly_observed = historical.groupby(["product_id", "week_start"]).quantity.sum()
grid = pd.MultiIndex.from_product(
    [products.product_id.tolist(), weeks], names=["product_id", "week_start"]
)
weekly = weekly_observed.reindex(grid, fill_value=0).rename("units_sold").reset_index()
first_seen = historical.groupby("product_id").week_start.min()
weekly = weekly.loc[weekly.week_start >= weekly.product_id.map(first_seen)].copy()
weekly.sort_values(["product_id", "week_start"], inplace=True)
weekly.reset_index(drop=True, inplace=True)
print(f"{len(weekly):,} produk-minggu dari {weekly.week_start.min().date()} sampai {weekly.week_start.max().date()}")
print("Porsi baris dengan penjualan tercatat nol:", round((weekly.units_sold == 0).mean(), 3))

## 4. Baseline prediksi mingguan

Prediksi untuk suatu minggu hanya memakai minggu-minggu sebelumnya. Kita bandingkan tiga aturan: selalu nol, hasil minggu lalu, dan rata-rata empat minggu sebelumnya. Ukur pada minggu mulai 2024-01-01 sampai akhir periode `imported`. Ini evaluasi **prediksi penjualan tercatat**, belum evaluasi keputusan diskon.

In [ ]:
by_product = weekly.groupby("product_id", sort=False).units_sold
weekly["lag_1"] = by_product.shift(1).fillna(0)
weekly["mean_4"] = by_product.transform(
    lambda series: series.shift(1).rolling(4, min_periods=1).mean()
).fillna(0)
evaluation = weekly.loc[weekly.week_start >= pd.Timestamp("2024-01-01")].copy()

def evaluate(actual, prediction):
    error = (actual - prediction).abs()
    return {
        "MAE": round(error.mean(), 3),
        "WAPE": round(error.sum() / actual.sum(), 3) if actual.sum() else np.nan,
        "MAE_when_sold": round(error.loc[actual > 0].mean(), 3),
    }

scores = pd.DataFrame({
    "always_zero": evaluate(evaluation.units_sold, np.zeros(len(evaluation))),
    "last_week": evaluate(evaluation.units_sold, evaluation.lag_1),
    "mean_previous_4_weeks": evaluate(evaluation.units_sold, evaluation.mean_4),
}).T
print(f"Evaluasi pada {len(evaluation):,} produk-minggu:")
print(scores.to_string())

Cara membaca metrik: banyak produk-minggu mencatat nol penjualan. Jika baseline always_zero mendapat MAE terbaik, itu menunjukkan target per produk-minggu sangat jarang dan model berikutnya perlu dibandingkan dengan baseline tersebut. Metrik ini tidak mengukur dampak diskon.


## 5. Contoh keputusan pada snapshot stok

Bagian ini memakai **empat minggu penuh terakhir** sebelum `stock_as_of_date`. Stok hanya dipakai pada tanggal snapshot itu. Aturan status berikut masih prototipe:

- `normal`: stok diperkirakan tidak berlebih untuk laju penjualan terakhir;
- `monitor`: stok tinggi atau kedaluwarsa dalam 90 hari;
- `discount`: kedaluwarsa dalam 30 hari, atau stok tinggi dengan kedaluwarsa dalam 90 hari.

Persentase diskon dibatasi agar harga setelah diskon tetap di atas biaya ditambah 5% dari harga normal. Ini **bukan** hasil model elastisitas harga.

In [ ]:
snapshot = products.stock_as_of_date.iloc[0]
assert products.stock_as_of_date.eq(snapshot).all()
last_day = snapshot - pd.Timedelta(days=1)
last_complete_sunday = last_day - pd.Timedelta(days=(last_day.dayofweek + 1) % 7)
window_start = last_complete_sunday - pd.Timedelta(days=27)
recent = transactions.loc[
    transactions.transaction_date.between(window_start, last_complete_sunday)
]
units_4w = recent.groupby("product_id").quantity.sum()
candidates = products.loc[
    (products.stock_on_hand > 0) & (products.expiry_date >= snapshot),
    ["product_id", "product_name", "category", "unit_cost", "list_price",
     "stock_on_hand", "expiry_date"]
].copy()
candidates["units_sold_last_4_weeks"] = candidates.product_id.map(units_4w).fillna(0).astype(int)
candidates["days_to_expiry"] = (candidates.expiry_date - snapshot).dt.days
candidates["weeks_of_cover"] = (
    candidates.stock_on_hand / (candidates.units_sold_last_4_weeks / 4).clip(lower=0.25)
).round(1)
print("Snapshot stok:", snapshot.date())
print("Empat minggu penuh:", window_start.date(), "s.d.", last_complete_sunday.date())
print("Produk aktif dengan stok:", len(candidates))

In [ ]:
def recommend(row):
    # High stock alone is monitored; expiry risk triggers a discount.
    high_stock = row.stock_on_hand >= 150 and row.weeks_of_cover >= 8
    should_discount = row.days_to_expiry <= 30 or (row.days_to_expiry <= 90 and high_stock)
    if should_discount:
        desired = 0.15 if row.days_to_expiry <= 14 else (0.10 if row.days_to_expiry <= 30 else 0.05)
        safe_rates = [
            rate for rate in (0.05, 0.10, 0.15)
            if rate <= desired and row.list_price * (1 - rate) >= row.unit_cost + 0.05 * row.list_price
        ]
        if safe_rates:
            reason = "expiry_soon" if row.days_to_expiry <= 30 else "high_stock_before_expiry"
            return "discount", max(safe_rates), reason
        return "monitor", 0.0, "margin_too_low_for_discount"
    if row.days_to_expiry <= 90 or high_stock:
        return "monitor", 0.0, "watch_stock_and_expiry"
    return "normal", 0.0, "no_near_term_discount_needed"

results = candidates.apply(recommend, axis=1, result_type="expand")
results.columns = ["status", "recommended_discount_rate", "reason"]
recommendations = pd.concat([candidates.reset_index(drop=True), results.reset_index(drop=True)], axis=1)
recommendations["recommendation_date"] = snapshot.date().isoformat()
recommendations["suggested_start_date"] = np.where(
    recommendations.status == "discount", snapshot.date().isoformat(), ""
)
print("Jumlah rekomendasi per status:")
print(recommendations.status.value_counts().to_string())
print("\nContoh produk prioritas:")
priority = {"discount": 0, "monitor": 1, "normal": 2}
top = recommendations.assign(priority=recommendations.status.map(priority)).sort_values(
    ["priority", "days_to_expiry", "weeks_of_cover"],
    ascending=[True, True, False],
).head(20)
print(top[[
    "product_id", "product_name", "stock_on_hand", "units_sold_last_4_weeks",
    "days_to_expiry", "status", "recommended_discount_rate", "reason"
]].to_string(index=False))

## Hasil dan langkah sesudah notebook ini

Simpan metrik baseline, periksa beberapa rekomendasi secara manual, lalu tentukan batas status bersama pemilik toko. Model regresi dapat ditambahkan setelah target, periode evaluasi, dan kualitas data disepakati. Evaluasi pada baris `generated` hanya berguna untuk menguji alur aplikasi karena data itu dibuat dari frekuensi penjualan periode awal.

Notebook ini tidak menulis balik ke tiga CSV sumber. Variabel `weekly`, `scores`, dan `recommendations` tersedia untuk eksperimen berikutnya.